In [2]:
!pip install mlflow


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


#### **IMPORT THE NECESSARY LIBRARIES**

In [3]:

import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import average_precision_score


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Setup the mlflow experiment**

In [4]:
mlflow.set_experiment("Assignment_02_SMS_Spam")


2026/03/04 00:09:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/04 00:09:34 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location=('file:///c:/Users/Arnab '
 'Bera/Desktop/CMI/CMI/Applied-ML/AppliedMachineLearning-main/AppliedMachineLearning/Assignment-02/mlruns/1'), creation_time=1771172939288, experiment_id='1', last_update_time=1771172939288, lifecycle_stage='active', name='Assignment_02_SMS_Spam', tags={}, workspace='default'>

#### **Load the dataset**

In [5]:
train_df = pd.read_csv("dataset/train.csv")
val_df = pd.read_csv("dataset/validation.csv")
test_df = pd.read_csv("dataset/test.csv")

X_train = train_df["message"]
y_train = train_df["label"]

X_val = val_df["message"]
y_val = val_df["label"]

X_test = test_df["message"]
y_test = test_df["label"]


#### **Text Vectorization**

In [6]:
vectorizer = TfidfVectorizer(stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)


#### **MODEDL-1: Logistic Regression**

In [13]:
rm -rf mlruns

In [15]:
import mlflow
import os

# Create fresh directory
os.makedirs("mlruns", exist_ok=True)

# Absolute path (safer than relative)
mlflow.set_tracking_uri("file://" + os.path.abspath("mlruns"))

mlflow.set_experiment("Assignment02_SMS_Spam")

with mlflow.start_run(run_name="Logistic_Regression"):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)

    probs = model.predict_proba(X_test_vec)[:, 1]
    aucpr = average_precision_score(y_test, probs)

    mlflow.log_param("model_name", "LogisticRegression")
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Logistic Regression AUCPR:", aucpr)

2026/03/04 00:38:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 00:38:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression AUCPR: 0.9646995083850685


#### **MODEL-2: NB**

In [16]:
with mlflow.start_run(run_name="Naive_Bayes"):

    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    probs = model.predict_proba(X_test_vec)[:, 1]
    aucpr = average_precision_score(y_test, probs)

    mlflow.log_param("model_name", "MultinomialNB")
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Naive Bayes AUCPR:", aucpr)


2026/03/04 00:38:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 00:38:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Naive Bayes AUCPR: 0.9786721665112575


#### **MODEL-3: Linear SVM**

In [17]:
with mlflow.start_run(run_name="Linear_SVM"):

    model = LinearSVC()
    model.fit(X_train_vec, y_train)

    scores = model.decision_function(X_test_vec)
    aucpr = average_precision_score(y_test, scores)

    mlflow.log_param("model_name", "LinearSVC")
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Linear SVM AUCPR:", aucpr)


2026/03/04 00:38:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 00:38:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear SVM AUCPR: 0.9851493315631116


#### **Show all the model**

In [19]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("Assignment02_SMS_Spam")

if experiment is None:
    print("Experiment not found!")
else:
    runs = client.search_runs([experiment.experiment_id])

    print("\nAll Model AUCPR Scores:\n")

    for run in runs:
        print(run.data.params.get("model_name"), 
              " : AUCPR:", 
              run.data.metrics.get("AUCPR"))


All Model AUCPR Scores:

LinearSVC  : AUCPR: 0.9851493315631116
MultinomialNB  : AUCPR: 0.9786721665112575
LogisticRegression  : AUCPR: 0.9646995083850685
None  : AUCPR: None
